# Voice Check-In with Speaker Verification
### MoveUp · HR AI prototype

A mover says:

> “I'm at the Ben Salah job, check me in.”

The phone also sends its GPS fix. The server must verify three things:

1. the voice matches the employee's previous enrollment,
2. the phrase contains the job and the **random challenge word** for this attempt,
3. the GPS fix is inside the client's geofence.

This notebook tests the full decision, including different-voice rejection,
replay resistance, and a realistic false-accept/false-reject analysis.

**All audio is real human speech. No TTS or synthetic voice is used.**


## 0. Kaggle setup

1. Enable **Internet** (Settings → Internet): the notebook downloads public
   human voice clips and the ECAPA model.
2. A GPU is optional. CPU works; a T4/P100 makes the embedding step faster.
3. Run the install cell once. `speechbrain` is not preinstalled on Kaggle;
   `librosa`, `soundfile`, `torch`, and `transformers` are.
4. If SpeechBrain cannot download or import, the notebook automatically falls
   back to the MFCC baseline. The check-in demo still runs.

Outputs go to `/kaggle/working/voice_output/`.


In [ ]:
%pip install -q "speechbrain>=1.0"

## 0. Imports and configuration

In [ ]:
import hashlib
import json
import math
import os
import random
import re
import shutil
import urllib.request
import warnings
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import display
from sklearn.metrics import auc, roc_curve

warnings.filterwarnings("ignore")

SEED = 42
rng = np.random.default_rng(SEED)
random.seed(SEED)

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT = WORK / "voice_output"
DATA = WORK / "voice_data"
SESSIONS = DATA / "sessions"
for folder in (OUT, DATA, SESSIONS):
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 150)

print("Output folder:", OUT)
print("Audio folder:", DATA)

## 1. Real human voice sources

The four `sv_speaker-*` files are the official speaker-verification demo:
**2 speakers × 2 sessions**. The other files are real human recordings from
the same public repository: JFK, MLK, a TED talk, French Common Voice and
French demo clips, plus Japanese, Russian, Bengali, and Afrikaans speakers.

That gives **13 source speakers / 26 sessions** without any TTS. Long recordings
are split into two halves so each speaker has an enrollment session and a test
session.

Sources and larger alternatives:

- https://huggingface.co/datasets/Xenova/transformers.js-docs
- VoxCeleb1/2 — large speaker-verification benchmark
- Mozilla Common Voice — consented, multilingual read speech
- LibriSpeech — read English speech

The single-file speakers are distinct source recordings, but their true
identities are not annotated. Treat the speaker labels for those files as
source-level labels, not a forensic ground truth. For production, collect
consented check-in utterances from your own employees and calibrate on those.


In [ ]:
# These files come from the public Transformers.js demo assets on Hugging Face.
# They are real human recordings, not TTS. The 4 sv_speaker files are the
# official speaker-verification demo (2 speakers x 2 sessions).
# For production, replace/extend with VoxCeleb, Common Voice, or your own
# consented check-in recordings.

HF_BASE = "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main"

PAIRED_SPEAKERS = {
    "SV-S1": ["sv_speaker-1_1.wav", "sv_speaker-1_2.wav"],
    "SV-S2": ["sv_speaker-2_1.wav", "sv_speaker-2_2.wav"],
}

SINGLE_SPEAKERS = {
    "JFK": ("jfk.wav", "en"),
    "MLK": ("mlk.wav", "en"),
    "TED": ("ted_60_16k.wav", "en"),
    "FR-COMMON": ("common_voice-fr.mp3", "fr"),
    "FR-AUDIO": ("french-audio.wav", "fr"),
    "FR-QUOTE": ("french-quote.wav", "fr"),
    "FR-COHERE": ("cohere_asr-fr.wav", "fr"),
    "JA": ("japanese-audio.wav", "ja"),
    "RU": ("russian-audio.wav", "ru"),
    "BN": ("bengali-audio.wav", "bn"),
    "AF": ("afrikaans-audio.wav", "af"),
}

EXPECTED_SHA256 = {
    "sv_speaker-1_1.wav": "c65258289d72485624a5ea06b0defd2c4012e672ed83198a327c3f5fc6a1f42b",
    "sv_speaker-1_2.wav": "a4e032b319a4ac5933a29a090ad23431f334528564eb381017da2692e56ec45f",
    "sv_speaker-2_1.wav": "20d3a722fecc2090ba5bc90fdfe7e4f3863b3695247b5ffc25f047a7e25d077d",
    "sv_speaker-2_2.wav": "9a2e91feae8d4b39a1c07572b852ccdec38707a0b8b13d3159cf957304a4eaf6",
    "jfk.wav": "aa81c2552465568567e670f3823117e633900d16bd6202346a72f3c8464c74c8",
}


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download(filename):
    target = DATA / filename
    if target.exists() and target.stat().st_size > 1_000:
        return target
    url = f"{HF_BASE}/{filename}"
    try:
        with urllib.request.urlopen(url, timeout=90) as response, open(target, "wb") as handle:
            shutil.copyfileobj(response, handle)
    except Exception as exc:
        raise RuntimeError(
            f"Could not download {filename}. Enable Internet in Kaggle "
            f"(Settings → Internet) or upload the files to {DATA}.\n{exc}"
        ) from exc
    return target


all_files = [name for names in PAIRED_SPEAKERS.values() for name in names]
all_files += [name for name, language in SINGLE_SPEAKERS.values()]

print("Downloading real human voice samples (no TTS)…")
for filename in all_files:
    path = download(filename)
    expected = EXPECTED_SHA256.get(filename)
    status = "ok"
    if expected and sha256(path) != expected:
        status = "CHECKSUM MISMATCH"
    print(f"  {filename:28s} {path.stat().st_size/1024:8.1f} KB  {status}")

print("\nSources:")
print("  https://huggingface.co/datasets/Xenova/transformers.js-docs")
print("  Larger options: VoxCeleb1/2 (speaker verification), Mozilla Common Voice (fr/en), LibriSpeech.")

## 2. Two sessions per speaker

Every clip is resampled to 16 kHz, silence-trimmed, peak-normalised, and capped
at 8 seconds. The two `sv_speaker` sessions are separate recordings; the other
speakers' two sessions are the first and second half of one real recording.
That makes the same-speaker task easier than true cross-device verification,
which is why the notebook also reports a noise/gain robustness check.


In [ ]:
TARGET_SR = 16_000
MAX_SECONDS = 8.0  # keep verification fast and comparable

records = []
audio = {}

def load_mono(path, sr=TARGET_SR):
    y, _ = librosa.load(path, sr=sr, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)
    if len(y) < int(0.5 * sr):
        raise ValueError(f"clip too short: {path}")
    y = y[: int(MAX_SECONDS * sr)]
    peak = np.max(np.abs(y)) or 1.0
    return (y / peak).astype(np.float32)


for speaker, names in PAIRED_SPEAKERS.items():
    for index, filename in enumerate(names):
        y = load_mono(DATA / filename)
        key = (speaker, index)
        audio[key] = y
        session_path = SESSIONS / f"{speaker}_s{index + 1}.wav"
        sf.write(session_path, y, TARGET_SR)
        records.append(
            {
                "speaker_id": speaker,
                "session": index,
                "filename": filename,
                "path": str(session_path),
                "duration_s": len(y) / TARGET_SR,
                "language": "en",
                "source": "speaker-verification demo",
            }
        )

for speaker, (filename, language) in SINGLE_SPEAKERS.items():
    y = load_mono(DATA / filename)
    # Split one continuous recording into two "sessions" (different halves).
    midpoint = len(y) // 2
    halves = [y[:midpoint], y[midpoint:]]
    for index, half in enumerate(halves):
        if len(half) < int(0.8 * TARGET_SR):
            continue
        key = (speaker, index)
        audio[key] = half
        session_path = SESSIONS / f"{speaker}_s{index + 1}.wav"
        sf.write(session_path, half, TARGET_SR)
        records.append(
            {
                "speaker_id": speaker,
                "session": index,
                "filename": filename,
                "path": str(session_path),
                "duration_s": len(half) / TARGET_SR,
                "language": language,
                "source": "public human recording (split)",
            }
        )

sessions = pd.DataFrame(records).sort_values(["speaker_id", "session"]).reset_index(drop=True)
print(f"Speakers: {sessions.speaker_id.nunique()} | sessions: {len(sessions)} | "
      f"languages: {', '.join(sorted(sessions.language.unique()))}")
display(sessions.groupby("speaker_id").agg(
    language=("language", "first"),
    sessions=("session", "count"),
    seconds=("duration_s", "sum"),
).reset_index())

## 3. Quick EDA

Waveforms and log-mel spectrograms show that the speakers differ in timbre,
not just loudness. The signal descriptors are only a sanity check; the actual
decision uses learned speaker embeddings.


In [ ]:
show_keys = [("SV-S1", 0), ("SV-S2", 0), ("JFK", 0), ("FR-COMMON", 0)]
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for ax, (speaker, index) in zip(axes.ravel(), show_keys):
    y = audio[(speaker, index)]
    ax.plot(np.arange(len(y)) / TARGET_SR, y, color="#111111", linewidth=0.6)
    ax.set_title(f"{speaker} · session {index + 1} · {len(y)/TARGET_SR:.1f}s", fontsize=10)
    ax.set_xlabel("seconds")
    ax.set_yticks([])
    ax.spines[["top", "right", "left"]].set_visible(False)
fig.suptitle("Real human speech used in this notebook — four different speakers", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "voice_waveforms.png", dpi=160)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for ax, (speaker, index) in zip(axes.ravel(), show_keys):
    y = audio[(speaker, index)]
    mel = librosa.feature.melspectrogram(y=y, sr=TARGET_SR, n_fft=400, hop_length=160, n_mels=64)
    db = librosa.power_to_db(mel, ref=np.max)
    image = librosa.display.specshow(db, sr=TARGET_SR, hop_length=160, x_axis="time", y_axis="mel", ax=ax, cmap="magma")
    ax.set_title(f"{speaker} · log-mel", fontsize=10)
cax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(image, cax=cax, format="%+2.0f dB")
fig.suptitle("Timbre differs across speakers - the cue used for voice matching", fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.9, 1])
fig.savefig(OUT / "voice_spectrograms.png", dpi=160, bbox_inches="tight")
plt.show()

feature_rows = []
for (speaker, index), y in audio.items():
    feature_rows.append(
        {
            "speaker_id": speaker,
            "session": index,
            "duration_s": round(len(y) / TARGET_SR, 2),
            "rms": round(float(np.sqrt(np.mean(y ** 2))), 4),
            "zcr": round(float(np.mean(librosa.feature.zero_crossing_rate(y))), 4),
            "centroid_hz": round(float(np.mean(librosa.feature.spectral_centroid(y=y, sr=TARGET_SR))), 1),
            "rolloff_hz": round(float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=TARGET_SR, roll_percent=0.85))), 1),
        }
    )
features_df = pd.DataFrame(feature_rows)
print("Signal descriptors")
display(features_df.head(12))

## 4. MFCC baseline

A light, interpretable baseline: mean MFCC coefficients C1–C19, L2-normalised,
compared with cosine similarity. C0 is dropped because it mostly encodes
loudness/energy and otherwise dominates the cosine score. This baseline is the
fallback if the deep model is unavailable.


In [ ]:
def mfcc_embedding(y, sr=TARGET_SR):
    """Mean of MFCC coefficients (C0 dropped), L2-normalised.

    Dropping C0 removes the loudness/energy component that otherwise dominates
    cosine similarity. On this speaker set the mean-only C1–C19 vector gave
    EER 0.064 vs 0.096 for the mean/std/delta version.
    """
    y = librosa.util.normalize(y)
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=20, n_fft=400, hop_length=160, n_mels=40, fmin=50, fmax=7600
    )
    vector = mfcc[1:].mean(axis=1)
    return (vector / (np.linalg.norm(vector) + 1e-9)).astype(np.float32)


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


mfcc_embeddings = {key: mfcc_embedding(y) for key, y in audio.items()}
print("MFCC embedding size:", next(iter(mfcc_embeddings.values())).shape)

# The 4 official speaker-verification clips: 2x2 similarity matrix.
sv_keys = [("SV-S1", 0), ("SV-S1", 1), ("SV-S2", 0), ("SV-S2", 1)]
sv_labels = ["S1-1", "S1-2", "S2-1", "S2-2"]
sv_matrix = np.array([[cosine(mfcc_embeddings[a], mfcc_embeddings[b]) for b in sv_keys] for a in sv_keys])

fig, ax = plt.subplots(figsize=(5.5, 4.6))
image = ax.imshow(sv_matrix, vmin=0, vmax=1, cmap="Greens")
ax.set_xticks(range(len(sv_labels)))
ax.set_xticklabels(sv_labels)
ax.set_yticks(range(len(sv_labels)))
ax.set_yticklabels(sv_labels)
for i in range(len(sv_labels)):
    for j in range(len(sv_labels)):
        ax.text(j, i, f"{sv_matrix[i, j]:.2f}", ha="center", va="center", fontsize=10)
ax.set_title("MFCC cosine similarity — real human speakers")
fig.colorbar(image, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(OUT / "voice_sv_matrix_mfcc.png", dpi=160)
plt.show()

print("Same speaker (S1-1 vs S1-2):", round(sv_matrix[0, 1], 3))
print("Different speaker (S1-1 vs S2-1):", round(sv_matrix[0, 2], 3))

## 5. Evaluation: same vs different speaker

Protocol: enroll session 1, test session 2 for every speaker. Same-speaker
trials are the 13 matching pairs; different-speaker trials are every test
session against every other speaker's enrollment.

Reported metrics:

- **EER** — equal error rate (FAR = FRR)
- **AUC** — ranking quality
- **accuracy** at the EER threshold
- **FAR / FRR** — false accept / false reject rates

This is a small public-set demonstration. Real deployment needs hundreds of
speakers, multiple devices, and domain check-in phrases.


In [ ]:
def trial_scores(embeddings):
    """Enroll session 1, test session 2, for every speaker."""
    speakers = sorted({speaker for speaker, _ in embeddings})
    same_scores = []
    diff_scores = []
    rows = []

    for speaker in speakers:
        if (speaker, 0) not in embeddings or (speaker, 1) not in embeddings:
            continue
        probe = embeddings[(speaker, 1)]
        same_scores.append(cosine(probe, embeddings[(speaker, 0)]))
        rows.append({"trial": "same", "speaker": speaker, "score": same_scores[-1]})

        for other in speakers:
            if other == speaker or (other, 0) not in embeddings:
                continue
            diff_scores.append(cosine(probe, embeddings[(other, 0)]))
            rows.append({"trial": "different", "speaker": speaker, "other": other, "score": diff_scores[-1]})

    return np.array(same_scores), np.array(diff_scores), pd.DataFrame(rows)


def eer_and_threshold(same_scores, diff_scores):
    labels = np.concatenate([np.ones(len(same_scores)), np.zeros(len(diff_scores))])
    scores = np.concatenate([same_scores, diff_scores])
    fpr, tpr, thresholds = roc_curve(labels, scores)
    fnr = 1 - tpr
    index = int(np.nanargmin(np.abs(fnr - fpr)))
    eer = float((fpr[index] + fnr[index]) / 2)
    threshold = float(thresholds[index])
    if not np.isfinite(threshold):
        threshold = float(np.median(scores))
    return {
        "eer": eer,
        "threshold": threshold,
        "auc": float(auc(fpr, tpr)),
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
    }


def decision_metrics(same_scores, diff_scores, threshold):
    far = float(np.mean(diff_scores >= threshold))
    frr = float(np.mean(same_scores < threshold))
    accuracy = float((np.sum(same_scores >= threshold) + np.sum(diff_scores < threshold))
                     / (len(same_scores) + len(diff_scores)))
    return {"far": far, "frr": frr, "accuracy": accuracy}


mfcc_same, mfcc_diff, mfcc_trials = trial_scores(mfcc_embeddings)
mfcc_eer = eer_and_threshold(mfcc_same, mfcc_diff)
mfcc_metrics = decision_metrics(mfcc_same, mfcc_diff, mfcc_eer["threshold"])

print(f"MFCC trials: {len(mfcc_same)} same, {len(mfcc_diff)} different")
print(f"EER={mfcc_eer['eer']:.3f} | threshold={mfcc_eer['threshold']:.3f} | "
      f"AUC={mfcc_eer['auc']:.3f} | accuracy={mfcc_metrics['accuracy']:.3f}")

fig, ax = plt.subplots(figsize=(9, 3.8))
bins = np.linspace(0, 1, 40)
ax.hist(mfcc_diff, bins=bins, alpha=0.65, label="different speaker", color="#bbbbbb", density=True)
ax.hist(mfcc_same, bins=bins, alpha=0.65, label="same speaker", color="#2e7d32", density=True)
ax.axvline(mfcc_eer["threshold"], color="#111", linestyle="--", linewidth=1.2, label=f"EER threshold {mfcc_eer['threshold']:.2f}")
ax.set_xlabel("Cosine similarity")
ax.set_ylabel("Density")
ax.set_title("MFCC baseline — same vs different speaker")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT / "voice_score_distribution_mfcc.png", dpi=160)
plt.show()

## 6. Deep speaker embeddings: ECAPA-TDNN

SpeechBrain's `spkrec-ecapa-voxceleb` maps each utterance to a 192-dimensional
embedding. It is trained for speaker recognition and is markedly stronger than
the MFCC baseline. The notebook downloads it from Hugging Face and falls back
to MFCC if anything fails.

We keep the same trial protocol so the two models are directly comparable.


In [ ]:
RUN_DEEP = True
DEEP_ERROR = None
device = "cpu"

try:
    import torch
    from speechbrain.inference.speaker import EncoderClassifier

    device = "cuda" if torch.cuda.is_available() else "cpu"
    classifier = EncoderClassifier.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir=str(WORK / "pretrained_models" / "spkrec-ecapa-voxceleb"),
        run_opts={"device": device},
    )
    print("ECAPA model loaded on", device)
except Exception as exc:
    RUN_DEEP = False
    DEEP_ERROR = str(exc)
    print("ECAPA unavailable — falling back to MFCC.")
    print(DEEP_ERROR)


def ecapa_embedding(y, sr=TARGET_SR):
    wav = torch.tensor(y, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = classifier.encode_batch(wav).squeeze().detach().cpu().numpy().astype(np.float32)
    return embedding / (np.linalg.norm(embedding) + 1e-9)


deep_embeddings = {}
if RUN_DEEP:
    for index, (key, y) in enumerate(audio.items(), start=1):
        deep_embeddings[key] = ecapa_embedding(y)
        if index % 5 == 0 or index == len(audio):
            print(f"  embedded {index}/{len(audio)}")
    print("ECAPA embedding size:", next(iter(deep_embeddings.values())).shape)

    deep_same, deep_diff, deep_trials = trial_scores(deep_embeddings)
    deep_eer = eer_and_threshold(deep_same, deep_diff)
    deep_metrics = decision_metrics(deep_same, deep_diff, deep_eer["threshold"])
    print(f"ECAPA: EER={deep_eer['eer']:.3f} | threshold={deep_eer['threshold']:.3f} | "
          f"AUC={deep_eer['auc']:.3f} | accuracy={deep_metrics['accuracy']:.3f}")

    deep_matrix = np.array([[cosine(deep_embeddings[a], deep_embeddings[b]) for b in sv_keys] for a in sv_keys])
    fig, ax = plt.subplots(figsize=(5.5, 4.6))
    image = ax.imshow(deep_matrix, vmin=0, vmax=1, cmap="Greens")
    ax.set_xticks(range(len(sv_labels)))
    ax.set_xticklabels(sv_labels)
    ax.set_yticks(range(len(sv_labels)))
    ax.set_yticklabels(sv_labels)
    for i in range(len(sv_labels)):
        for j in range(len(sv_labels)):
            ax.text(j, i, f"{deep_matrix[i, j]:.2f}", ha="center", va="center", fontsize=10)
    ax.set_title("ECAPA cosine similarity — real human speakers")
    fig.colorbar(image, ax=ax, shrink=0.8)
    fig.tight_layout()
    fig.savefig(OUT / "voice_sv_matrix_ecapa.png", dpi=160)
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 3.8))
    bins = np.linspace(-0.2, 1.0, 50)
    ax.hist(deep_diff, bins=bins, alpha=0.65, label="different speaker", color="#bbbbbb", density=True)
    ax.hist(deep_same, bins=bins, alpha=0.65, label="same speaker", color="#2e7d32", density=True)
    ax.axvline(deep_eer["threshold"], color="#111", linestyle="--", linewidth=1.2,
               label=f"EER threshold {deep_eer['threshold']:.2f}")
    ax.set_xlabel("Cosine similarity")
    ax.set_ylabel("Density")
    ax.set_title("ECAPA-TDNN — same vs different speaker")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(OUT / "voice_score_distribution_ecapa.png", dpi=160)
    plt.show()

    np.savez_compressed(
        OUT / "ecapa_embeddings.npz",
        **{f"{speaker}__{session}": vector for (speaker, session), vector in deep_embeddings.items()},
    )

# Comparison table / chart.
comparison_rows = [
    {
        "model": "MFCC + cosine",
        "embedding_dim": len(next(iter(mfcc_embeddings.values()))),
        "eer": round(mfcc_eer["eer"], 3),
        "threshold": round(mfcc_eer["threshold"], 3),
        "auc": round(mfcc_eer["auc"], 3),
        "accuracy": round(mfcc_metrics["accuracy"], 3),
    }
]
if RUN_DEEP:
    comparison_rows.append(
        {
            "model": "ECAPA-TDNN",
            "embedding_dim": len(next(iter(deep_embeddings.values()))),
            "eer": round(deep_eer["eer"], 3),
            "threshold": round(deep_eer["threshold"], 3),
            "auc": round(deep_eer["auc"], 3),
            "accuracy": round(deep_metrics["accuracy"], 3),
        }
    )
comparison = pd.DataFrame(comparison_rows)
print("Speaker-verification comparison")
display(comparison.style.hide(axis="index"))

fig, ax = plt.subplots(figsize=(7, 3.6))
x = np.arange(len(comparison))
ax.bar(x - 0.18, comparison.eer, 0.36, label="EER (lower is better)", color="#e61919")
ax.bar(x + 0.18, 1 - comparison.accuracy, 0.36, label="error rate (lower is better)", color="#111111")
ax.set_xticks(x)
ax.set_xticklabels(comparison.model)
ax.set_ylim(0, max(0.5, float(comparison.eer.max()) + 0.1))
ax.set_title("Speaker verification on real human recordings")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT / "voice_model_comparison.png", dpi=160)
plt.show()

## 7. Voice check-in policy

The server-side decision combines:

- voice similarity (with an **accept / review / reject** band),
- the parsed check-in intent,
- the job named in the phrase,
- the random challenge word for this attempt,
- the GPS distance to the client's geofence.

The scenario table uses **real audio** for the speaker score. The transcripts
are scripted strings for the intent/geofence tests; the public clips are not
literal moving-company commands. In production the ASR transcript feeds the
same parser.

A replay of the enrollment file scores 1.0 for voice, which is exactly why the
challenge word and a liveness check are required.


In [ ]:
JOBS = [
    {"job_id": "JOB-101", "client": "Ben Salah", "area": "La Marsa", "lat": 36.8782, "lng": 10.3247},
    {"job_id": "JOB-102", "client": "Trabelsi", "area": "Carthage", "lat": 36.8528, "lng": 10.3233},
    {"job_id": "JOB-103", "client": "Gharbi", "area": "Tunis Centre", "lat": 36.8065, "lng": 10.1815},
    {"job_id": "JOB-104", "client": "Mansour", "area": "Ariana", "lat": 36.8625, "lng": 10.1956},
]
JOBS_BY_ID = {job["job_id"]: job for job in JOBS}

CHALLENGE_WORDS = ["Tunis", "Sfax", "Bizerte", "Sousse", "Nabeul", "Kairouan"]
GEOFENCE_M = 300
REVIEW_MARGIN = 0.03 if RUN_DEEP else 0.0
MIN_SPEECH_S = 1.0

active_model = "ecapa" if RUN_DEEP else "mfcc"
active_embeddings = deep_embeddings if RUN_DEEP else mfcc_embeddings
active_threshold = float(deep_eer["threshold"]) if RUN_DEEP else float(mfcc_eer["threshold"])
active_eer = float(deep_eer["eer"]) if RUN_DEEP else float(mfcc_eer["eer"])
active_auc = float(deep_eer["auc"]) if RUN_DEEP else float(mfcc_eer["auc"])

print(f"Active engine: {active_model} | threshold={active_threshold:.3f} | EER={active_eer:.3f}")


def haversine_m(lat1, lng1, lat2, lng2):
    radius = 6_371_000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lng2 - lng1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * radius * math.asin(math.sqrt(a))


def normalize_text(text):
    return " ".join(
        text.lower()
        .replace("’", "'")
        .replace("-", " ")
        .replace(",", " ")
        .replace(".", " ")
        .split()
    )


def parse_command(transcript):
    """Small deterministic intent parser. Replace with an LLM tool call if needed."""
    text = normalize_text(transcript)
    intent = "unknown"
    if any(phrase in text for phrase in ["check me in", "check in", "checked in", "i'm at", "i am at", "je suis", "arrivé", "arrive"]):
        intent = "check_in"

    job = None
    for candidate in JOBS:
        if candidate["client"].lower() in text or candidate["area"].lower() in text:
            job = candidate
            break

    # Prefer an explicit "code <word>" challenge. Otherwise remove job names/areas
    # before searching, so "Tunis Centre" is not mistaken for challenge "Tunis".
    challenge = None
    code_match = re.search(r"\bcode:?\s+([a-z\u00e9\u00e8\u00ea\u00e0\u00e2\u00ee\u00f4\u00fb\u00e7]+)", text)
    if code_match:
        candidate_word = code_match.group(1).capitalize()
        if candidate_word in CHALLENGE_WORDS:
            challenge = candidate_word
    if challenge is None:
        challenge_text = text
        for candidate in JOBS:
            challenge_text = challenge_text.replace(candidate["client"].lower(), " ")
            challenge_text = challenge_text.replace(candidate["area"].lower(), " ")
        for word in CHALLENGE_WORDS:
            if word.lower() in challenge_text:
                challenge = word
                break

    return {"intent": intent, "job": job, "challenge": challenge, "text": text}


def verify_voice(probe_audio, enrolled_audio, threshold):
    """Return the similarity score and a 3-way decision."""
    if len(probe_audio) / TARGET_SR < MIN_SPEECH_S:
        return 0.0, "low_quality"
    if RUN_DEEP:
        probe_vec = ecapa_embedding(probe_audio)
        enroll_vec = ecapa_embedding(enrolled_audio)
    else:
        probe_vec = mfcc_embedding(probe_audio)
        enroll_vec = mfcc_embedding(enrolled_audio)
    score = cosine(probe_vec, enroll_vec)
    if score < threshold - REVIEW_MARGIN:
        label = "different_voice"
    elif score < threshold + REVIEW_MARGIN:
        label = "review"
    else:
        label = "same_voice"
    return score, label


def decide_checkin(probe_audio, enrolled_audio, transcript, expected_job_id, expected_challenge, checkin_lat, checkin_lng):
    parsed = parse_command(transcript)
    job = JOBS_BY_ID[expected_job_id]
    distance = haversine_m(checkin_lat, checkin_lng, job["lat"], job["lng"])
    score, voice_label = verify_voice(probe_audio, enrolled_audio, active_threshold)

    reasons = []
    if parsed["intent"] != "check_in":
        reasons.append("no_check_in_intent")
    if voice_label == "different_voice":
        reasons.append("different_voice")
    if voice_label == "review":
        reasons.append("voice_needs_review")
    if parsed["challenge"] is None:
        reasons.append("missing_challenge")
    elif parsed["challenge"].lower() != expected_challenge.lower():
        reasons.append("wrong_challenge")
    if parsed["job"] is None:
        reasons.append("job_not_recognised")
    elif parsed["job"]["job_id"] != expected_job_id:
        reasons.append("wrong_job")
    if distance > GEOFENCE_M:
        reasons.append("outside_geofence")

    hard_reasons = [reason for reason in reasons if reason != "voice_needs_review"]
    if hard_reasons:
        decision = "reject"
    elif "voice_needs_review" in reasons:
        decision = "review"
    else:
        decision = "accept"

    return {
        "decision": decision,
        "reasons": reasons,
        "speaker_score": round(score, 4),
        "voice_label": voice_label,
        "threshold": round(active_threshold, 4),
        "challenge_ok": parsed["challenge"] is not None and parsed["challenge"].lower() == expected_challenge.lower(),
        "job_ok": parsed["job"] is not None and parsed["job"]["job_id"] == expected_job_id,
        "distance_m": round(distance, 1),
        "parsed": parsed,
    }


enrolled_voice = audio[("SV-S1", 0)]  # Alice's enrollment recording
probe_same = audio[("SV-S1", 1)]      # Alice, another session
probe_other = audio[("SV-S2", 1)]     # Bob, different voice
job_101 = JOBS_BY_ID["JOB-101"]
near_job = (job_101["lat"] + 0.0002, job_101["lng"] + 0.0002)
far_from_job = (job_101["lat"] + 0.02, job_101["lng"] + 0.02)  # ~2.8 km

scenarios = [
    {
        "scenario": "same speaker, correct job + challenge + GPS",
        "probe": probe_same,
        "transcript": "I'm at the Ben Salah job in La Marsa, code Tunis, check me in",
        "expected_job": "JOB-101",
        "challenge": "Tunis",
        "gps": near_job,
        "expected": "accept",
    },
    {
        "scenario": "different speaker (buddy punch)",
        "probe": probe_other,
        "transcript": "I'm at the Ben Salah job in La Marsa, code Tunis, check me in",
        "expected_job": "JOB-101",
        "challenge": "Tunis",
        "gps": near_job,
        "expected": "reject",
    },
    {
        "scenario": "same speaker, missing challenge (replay)",
        "probe": probe_same,
        "transcript": "I'm at the Ben Salah job in La Marsa, check me in",
        "expected_job": "JOB-101",
        "challenge": "Tunis",
        "gps": near_job,
        "expected": "reject",
    },
    {
        "scenario": "same speaker, wrong job",
        "probe": probe_same,
        "transcript": "I'm at the Trabelsi job in Carthage, code Tunis, check me in",
        "expected_job": "JOB-101",
        "challenge": "Tunis",
        "gps": near_job,
        "expected": "reject",
    },
    {
        "scenario": "same speaker, outside geofence",
        "probe": probe_same,
        "transcript": "I'm at the Ben Salah job in La Marsa, code Tunis, check me in",
        "expected_job": "JOB-101",
        "challenge": "Tunis",
        "gps": far_from_job,
        "expected": "reject",
    },
    {
        "scenario": "exact replay of enrollment audio, fresh challenge",
        "probe": audio[("SV-S1", 0)],
        "transcript": "I'm at the Ben Salah job in La Marsa, code Tunis, check me in",
        "expected_job": "JOB-101",
        "challenge": "Sfax",
        "gps": near_job,
        "expected": "reject",
    },
    {
        "scenario": "same speaker, wrong challenge",
        "probe": probe_same,
        "transcript": "I'm at the Ben Salah job in La Marsa, code Sfax, check me in",
        "expected_job": "JOB-101",
        "challenge": "Tunis",
        "gps": near_job,
        "expected": "reject",
    },
    {
        "scenario": "off-topic speech (no check-in intent)",
        "probe": audio[("JFK", 0)],
        "transcript": "Ask not what your country can do for you",
        "expected_job": "JOB-101",
        "challenge": "Tunis",
        "gps": near_job,
        "expected": "reject",
    },
]

scenario_rows = []
for scenario in scenarios:
    result = decide_checkin(
        scenario["probe"],
        enrolled_voice,
        scenario["transcript"],
        scenario["expected_job"],
        scenario["challenge"],
        *scenario["gps"],
    )
    scenario_rows.append(
        {
            "scenario": scenario["scenario"],
            "speaker_score": result["speaker_score"],
            "voice": result["voice_label"],
            "challenge_ok": result["challenge_ok"],
            "job_ok": result["job_ok"],
            "distance_m": result["distance_m"],
            "decision": result["decision"],
            "reasons": ", ".join(result["reasons"]) or "—",
            "expected": scenario["expected"],
            "pass": result["decision"] == scenario["expected"],
        }
    )

scenario_df = pd.DataFrame(scenario_rows)
print("Voice check-in scenario tests")
display(scenario_df.style.hide(axis="index"))

## 7b. Robustness: real voice + channel noise

The probe is a real human recording. We add white noise at 15 dB SNR and a
−6 dB gain change to imitate a noisy truck cab or a weak phone microphone.
No synthetic speech is created; only the real recording is degraded.


In [ ]:
def add_white_noise(y, snr_db=15.0):
    signal_power = np.mean(y ** 2)
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = rng.normal(0, np.sqrt(noise_power), len(y))
    mixed = y + noise
    return (mixed / (np.max(np.abs(mixed)) + 1e-9)).astype(np.float32)


if RUN_DEEP:
    clean_score = cosine(ecapa_embedding(probe_same), ecapa_embedding(enrolled_voice))
    noisy_probe = add_white_noise(probe_same, snr_db=15.0)
    noisy_score = cosine(ecapa_embedding(noisy_probe), ecapa_embedding(enrolled_voice))
    gain_score = cosine(ecapa_embedding((probe_same * 0.5).astype(np.float32)), ecapa_embedding(enrolled_voice))
    robustness = pd.DataFrame(
        [
            {"condition": "clean", "score": round(clean_score, 4), "accept": clean_score >= active_threshold},
            {"condition": "15 dB white noise", "score": round(noisy_score, 4), "accept": noisy_score >= active_threshold},
            {"condition": "−6 dB gain", "score": round(gain_score, 4), "accept": gain_score >= active_threshold},
        ]
    )
    print("Robustness on a real human probe (noise/gain added to the real recording)")
    display(robustness.style.hide(axis="index"))

## 7c. Optional ASR: real audio → transcript

Whisper tiny is optional. It demonstrates the audio-to-text stage on real human
clips. The public clips are not check-in phrases, so the transcript will be
ordinary read speech; the parser is tested separately with realistic check-in
transcripts. If `transformers` or the model download is unavailable, this cell
skips cleanly.


In [ ]:
RUN_ASR = True
ASR_ERROR = None
asr = None
if RUN_ASR:
    try:
        from transformers import pipeline

        asr_device = 0 if (RUN_DEEP and device == "cuda") else -1
        asr = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-tiny",
            device=asr_device,
        )
        print("Whisper tiny loaded on", "cuda" if asr_device == 0 else "cpu")
    except Exception as exc:
        ASR_ERROR = str(exc)
        print("ASR unavailable:", ASR_ERROR)

if asr is not None:
    asr_rows = []
    for label, key in [("JFK (en)", ("JFK", 0)), ("SV-S1 (en)", ("SV-S1", 1)), ("FR-COMMON (fr)", ("FR-COMMON", 0))]:
        y = audio[key]
        # Pipeline accepts a float32 waveform at 16 kHz.
        try:
            result = asr(y, generate_kwargs={"language": "en" if key[0] != "FR-COMMON" else "fr"})
            transcript = result["text"].strip()
        except Exception as exc:
            transcript = f"[ASR error: {exc}]"
        asr_rows.append({"clip": label, "transcript": transcript})
    asr_df = pd.DataFrame(asr_rows)
    print("Real audio transcriptions (Whisper tiny)")
    display(asr_df.style.hide(axis="index"))
    print("Note: these public clips are not check-in commands. The parser is tested with realistic")
    print("check-in transcripts above; in production the same ASR feeds parse_command().")

## 8. Replay and liveness

An exact replay has waveform correlation 1.0 and voice score 1.0. Voice
matching alone cannot stop it. The defense is a **fresh random challenge word**
on every attempt, plus a server-side liveness/anti-spoofing step in production.
The table shows the difference between an exact replay, a genuine second
session, and a different speaker.


In [ ]:
def waveform_correlation(a, b):
    length = min(len(a), len(b))
    a = a[:length] - np.mean(a[:length])
    b = b[:length] - np.mean(b[:length])
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


replay_rows = [
    {
        "probe": "same enrollment file (exact replay)",
        "waveform_correlation": round(waveform_correlation(enrolled_voice, enrolled_voice), 4),
        "voice_score": round(cosine(active_embeddings[("SV-S1", 0)], active_embeddings[("SV-S1", 0)]), 4),
        "challenge_defence": "random challenge phrase changes every attempt",
    },
    {
        "probe": "real second session (live person)",
        "waveform_correlation": round(waveform_correlation(probe_same, enrolled_voice), 4),
        "voice_score": round(cosine(active_embeddings[("SV-S1", 1)], active_embeddings[("SV-S1", 0)]), 4),
        "challenge_defence": "correct random challenge required",
    },
    {
        "probe": "different speaker",
        "waveform_correlation": round(waveform_correlation(probe_other, enrolled_voice), 4),
        "voice_score": round(cosine(active_embeddings[("SV-S2", 1)], active_embeddings[("SV-S1", 0)]), 4),
        "challenge_defence": "voice mismatch rejected",
    },
]
replay_df = pd.DataFrame(replay_rows)
print("Replay / liveness analysis")
display(replay_df.style.hide(axis="index"))
print("A replayed recording can match the voice perfectly. That is why voice alone is not enough:")
print("the server must require a fresh random challenge phrase (and ideally anti-spoofing).")

## 9. Export policy and results

`voice_policy.json` is the contract for the Next.js app: active engine,
threshold, review margin, EER, geofence radius, challenge words, and privacy
flags. The CSV files contain the trial scores and scenario outcomes; the PNGs
are ready for a report or slide.

Store only the enrollment embedding, not raw audio. Require explicit consent,
encrypt the embedding, and provide a delete path.


In [ ]:
policy = {
    "version": 1,
    "engine": active_model,
    "embedding_model": "speechbrain/spkrec-ecapa-voxceleb" if RUN_DEEP else "MFCC (baseline)",
    "embedding_dim": int(len(next(iter(active_embeddings.values())))),
    "threshold": round(active_threshold, 4),
    "review_margin": REVIEW_MARGIN,
    "eer": round(active_eer, 4),
    "auc": round(active_auc, 4),
    "geofence_radius_m": GEOFENCE_M,
    "challenge_words": CHALLENGE_WORDS,
    "min_speech_s": MIN_SPEECH_S,
    "max_session_s": MAX_SECONDS,
    "decisions": ["accept", "review", "reject"],
    "reject_reasons": [
        "different_voice",
        "voice_needs_review",
        "missing_challenge",
        "wrong_challenge",
        "job_not_recognised",
        "wrong_job",
        "outside_geofence",
        "no_check_in_intent",
    ],
    "privacy": {
        "store_raw_audio": False,
        "store_embedding_only": True,
        "consent_required": True,
        "human_review_for_reject": True,
    },
}
with open(OUT / "voice_policy.json", "w", encoding="utf-8") as handle:
    json.dump(policy, handle, indent=2)

comparison.to_csv(OUT / "voice_model_comparison.csv", index=False)
scenario_df.to_csv(OUT / "voice_scenario_results.csv", index=False)
mfcc_trials.to_csv(OUT / "voice_mfcc_trials.csv", index=False)
if RUN_DEEP:
    deep_trials.to_csv(OUT / "voice_ecapa_trials.csv", index=False)
sessions.to_csv(OUT / "voice_sessions.csv", index=False)
replay_df.to_csv(OUT / "voice_replay_analysis.csv", index=False)

print("Exported files:")
for path in sorted(OUT.iterdir()):
    print(" -", path.name, f"({path.stat().st_size/1024:.1f} KB)")

print("\nVoice policy")
print(json.dumps(policy, indent=2))

## Production blueprint for MoveUp

**Enrollment** (`POST /api/voice/enroll`): record 3–5 consented utterances,
check quality/liveness, compute an ECAPA embedding server-side, store the
embedding and consent record. Never store raw audio by default.

**Check-in** (`POST /api/voice/checkin`): the server issues a random challenge
word; the phone records the phrase plus GPS; the server transcribes, checks the
job/challenge, computes the probe embedding, compares it to the enrolled
embedding, and checks the geofence. Return `accept`, `review`, or `reject` with
machine-readable reasons.

**Operations**: calibrate the threshold per device/language on consented data,
monitor FAR/FRR, route `review` cases to a human, and require human review
before any disciplinary action. Voice is biometric data: obtain consent,
minimise retention, and comply with applicable privacy law.


## 10. Live demo API via ngrok (Kaggle → MoveUp app)

The notebook above is batch analysis. To demo voice check-in from the Next.js app,
expose a tiny HTTP server from this session with ngrok:

1. Add a secret named `NGROK_AUTHTOKEN` in Kaggle (Settings → Add a secret) — token from https://dashboard.ngrok.com
2. Run the install cell below (needs Internet = ON).
3. Run the server cell — it prints a `https://…ngrok-free.app` URL.
4. In the Next.js app `.env`, set `VOICE_API_BASE_URL=<that URL>` and restart. The app then calls `POST /api/voice/checkin` on the tunnel.

Endpoints: `GET /health`, `POST /api/voice/checkin` (JSON: audio_b64, transcript, GPS, job, challenge).
**Never commit the ngrok token.** The tunnel dies when the session ends.

In [ ]:
%pip install -q ngrok flask

import base64, io, threading
from flask import Flask, jsonify, request
import soundfile as sf

app = Flask(__name__)

@app.get("/health")
def health():
    return jsonify({"ok": True, "engine": active_model, "threshold": active_threshold})

@app.post("/api/voice/checkin")
def voice_checkin():
    payload = request.get_json(force=True)
    y, _ = sf.read(io.BytesIO(base64.b64decode(payload.get("audio_b64", ""))), dtype="float32", always_2d=False)
    return jsonify(decide_checkin(np.asarray(y, dtype=np.float32), enrolled_voice, payload.get("transcript", ""), payload.get("job_id", "JOB-101"), payload.get("challenge", "Tunis"), float(payload.get("lat", near_job[0])), float(payload.get("lng", near_job[1]))))

PORT = 8001
threading.Thread(target=lambda: app.run(host="0.0.0.0", port=PORT, use_reloader=False), daemon=True).start()
print(f"Local server on http://0.0.0.0:{PORT}")

NGROK_TOKEN = "<NGROK_AUTHTOKEN>"
NGROK_DOMAIN = "paloma-cranelike-kathrin.ngrok-free.dev"
import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
conn = await ngrok.connect(str(PORT), domain=NGROK_DOMAIN)
print("Stable URL:")
print(conn.url())

**Static URL setup (one-time):** ngrok free gives you **1 static domain**. In dashboard.ngrok.com → **Cloud Edge → Domains → Create a static domain** (e.g. `moveup-voice.ngrok-free.app`). Add it as a Kaggle secret named `NGROK_DOMAIN` next to `NGROK_AUTHTOKEN`. From then on the printed URL is always the same — set `VOICE_API_BASE_URL` once in the Next.js `.env` and never change it.